# Explaining Stage-2 EV vs ICE — YOLO26s

Model explainability for the **best Stage-2 classifier** from `training_pipeline.ipynb`:

- **Model:** `yolo26s_cls` (exported under `models/best_ev_ice_classifier/`)
- **Data:** the same held-out **test split** (`SEED=42`, `TEST_SIZE=0.15`, `VAL_SIZE=0.20`)
- **Methods:** Grad-CAM, Grad-CAM++, EigenCAM, occlusion sensitivity

Goal: show what spatial cues the model uses for **EV vs ICE** predictions — useful figures and narrative for the paper.

Figures are written to `runs/explain_stage2_yolo26s/`.

## 1) Imports & configuration

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 200,
        "font.size": 11,
        "axes.titlesize": 11,
        "axes.titleweight": "semibold",
    }
)


In [ ]:
TRAINER_ROOT = Path.cwd().resolve()
DATASET_ROOT = TRAINER_ROOT / "dataset"
ANNOTATIONS_CSV = DATASET_ROOT / "annotations.csv"
RUNS_DIR = TRAINER_ROOT / "runs"
PRED_DIR = RUNS_DIR / "_significance_preds"
EXPORT_DIR = TRAINER_ROOT / "models" / "best_ev_ice_classifier"
EXPORT_META = EXPORT_DIR / "metadata.json"
WEIGHTS_PATH = EXPORT_DIR / "best.pt"

VALID_LABELS = ("ev", "ice")
CLASS_TO_IDX = {"ev": 0, "ice": 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
EV_INDEX = CLASS_TO_IDX["ev"]
ICE_INDEX = CLASS_TO_IDX["ice"]

SEED = 42
VAL_SIZE = 0.20
TEST_SIZE = 0.15
IMG_SIZE = 224

MODEL_NAME = "yolo26s_cls"
TARGET_LAYER_NAME = "model.9"

OUT_DIR = RUNS_DIR / "explain_stage2_yolo26s"
N_PER_BUCKET = 4
OCCLUSION_PATCH = 32
OCCLUSION_STRIDE = 16
SAVE_FIGURES = True

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Trainer root : {TRAINER_ROOT}")
print(f"Weights      : {WEIGHTS_PATH}")
print(f"Output dir   : {OUT_DIR}")
print(f"Model        : {MODEL_NAME}")


## 2) Reconstruct the held-out test set

Identical stratified split to `training_pipeline.ipynb` / `results_analysis.ipynb`.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_labeled_dataset(csv_path: Path, dataset_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    required = {"relative_path", "label"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"annotations.csv missing columns: {sorted(missing_cols)}")

    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(VALID_LABELS)].copy()
    df["image_path"] = df["relative_path"].map(lambda rel: dataset_root / rel)
    df["exists"] = df["image_path"].map(lambda p: Path(p).exists())

    missing = df[~df["exists"]]
    if len(missing):
        print(f"Dropping {len(missing)} rows with missing image files")

    df = df[df["exists"]].reset_index(drop=True)
    df["label_idx"] = df["label"].map(CLASS_TO_IDX).astype(int)
    return df

def split_dataframe(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df["label_idx"],
    )
    val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_ratio,
        random_state=SEED,
        stratify=train_df["label_idx"],
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

set_seed(SEED)
DEVICE = resolve_device()
print(f"Device: {DEVICE}")

labeled_df = load_labeled_dataset(ANNOTATIONS_CSV, DATASET_ROOT)
_, _, test_df = split_dataframe(labeled_df)

print(f"Usable labeled images: {len(labeled_df)}")
print(f"Test set size        : {len(test_df)}")
print(test_df["label"].value_counts())
test_df.head(3)


## 3) Load YOLO26s + test predictions

Prefer cached predictions from `runs/_significance_preds/yolo26s_cls_test_preds.json` (same order as `test_df`). Fall back to live inference if the cache is missing.

In [ ]:
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {WEIGHTS_PATH}. Export the best model from training_pipeline.ipynb first."
    )

meta = json.loads(EXPORT_META.read_text()) if EXPORT_META.exists() else {}
print(
    f"Export metadata: experiment={meta.get('exported_experiment')} "
    f"family={meta.get('family')} backbone={meta.get('backbone')} "
    f"test_f1_ev={meta.get('metrics', {}).get('test_f1_ev')}"
)

yolo = YOLO(str(WEIGHTS_PATH))
model = yolo.model.to(DEVICE)
model.eval()

names = getattr(yolo, "names", {0: "ev", 1: "ice"})
print(f"Class names: {names}")
print(f"Target layer for CAM: {TARGET_LAYER_NAME}")
print(model.model[9].__class__.__name__, "@", TARGET_LAYER_NAME)


In [ ]:
def load_or_run_predictions(test_df: pd.DataFrame) -> pd.DataFrame:
    cache_path = PRED_DIR / f"{MODEL_NAME}_test_preds.json"
    if cache_path.exists():
        cached = json.loads(cache_path.read_text())
        y_true = np.asarray(cached["y_true"], dtype=int)
        y_pred = np.asarray(cached["y_pred"], dtype=int)
        y_prob_ev = np.asarray(cached["y_prob_ev"], dtype=float)
        if len(y_true) != len(test_df):
            raise RuntimeError(f"Cache length {len(y_true)} != test_df {len(test_df)}")
        if not np.array_equal(y_true, test_df["label_idx"].to_numpy()):
            raise RuntimeError("Cached y_true does not match reconstructed test_df order")
        print(f"Loaded cached predictions: {cache_path.name}")
        print(
            f"  accuracy={cached.get('accuracy'):.4f}  "
            f"precision_ev={cached.get('precision_ev'):.4f}  "
            f"recall_ev={cached.get('recall_ev'):.4f}  "
            f"f1_ev={cached.get('f1_ev'):.4f}"
        )
    else:
        print("Cache missing — running live Ultralytics inference on test set…")
        y_true, y_pred, y_prob_ev = [], [], []
        for path, label_idx in zip(test_df["image_path"], test_df["label_idx"]):
            result = yolo.predict(str(path), imgsz=IMG_SIZE, verbose=False)[0]
            probs = result.probs.data.detach().cpu().numpy()
            pred_idx = int(result.probs.top1)
            y_true.append(int(label_idx))
            y_pred.append(pred_idx)
            y_prob_ev.append(float(probs[EV_INDEX]))
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)
        y_prob_ev = np.asarray(y_prob_ev, dtype=float)

    df = test_df.copy()
    df["y_true"] = y_true
    df["y_pred"] = y_pred
    df["prob_ev"] = y_prob_ev
    df["prob_ice"] = 1.0 - y_prob_ev
    df["gt_label"] = df["y_true"].map(IDX_TO_CLASS)
    df["pred_label"] = df["y_pred"].map(IDX_TO_CLASS)
    df["correct"] = df["y_true"] == df["y_pred"]
    df["confidence"] = np.where(df["y_pred"] == EV_INDEX, df["prob_ev"], df["prob_ice"])

    buckets = []
    for _, row in df.iterrows():
        if row["y_true"] == EV_INDEX and row["y_pred"] == EV_INDEX:
            buckets.append("TP_ev")
        elif row["y_true"] == ICE_INDEX and row["y_pred"] == ICE_INDEX:
            buckets.append("TN_ice")
        elif row["y_true"] == ICE_INDEX and row["y_pred"] == EV_INDEX:
            buckets.append("FP_ice_as_ev")
        else:
            buckets.append("FN_ev_as_ice")
    df["bucket"] = buckets
    return df

analysis_df = load_or_run_predictions(test_df)
print()
print("Outcome counts:")
display(analysis_df["bucket"].value_counts().rename("n").to_frame())
analysis_df[["relative_path", "gt_label", "pred_label", "prob_ev", "confidence", "bucket"]].head()


## 4) Explainability methods

All methods operate on the Ultralytics classification backbone with ImageNet-style preprocess (`classify_transforms`, 224×224).

| Method | Type | What it shows |
|---|---|---|
| **Grad-CAM** | gradient × activation | Class-specific regions supporting the chosen logit |
| **Grad-CAM++** | weighted positive gradients | Sharper localization, better for multiple objects |
| **EigenCAM** | PCA of activations | Gradient-free “where the features fire” |
| **Occlusion** | input perturbation | Drop in target-class score when a patch is masked |

Target layer defaults to **`model.9` (C2PSA)** — the last spatial block before the classify head.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TRANSFORMS = classify_transforms(IMG_SIZE, mean=IMAGENET_MEAN, std=IMAGENET_STD)

def get_module_by_name(root: torch.nn.Module, dotted: str) -> torch.nn.Module:
    mod: torch.nn.Module = root
    for part in dotted.split("."):
        if part.isdigit():
            mod = mod[int(part)]
        else:
            mod = getattr(mod, part)
    return mod

def load_rgb(path: str | Path) -> np.ndarray:
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def preprocess_rgb(rgb: np.ndarray) -> tuple[torch.Tensor, np.ndarray]:
    """Return (1,C,H,W) tensor and the resized/cropped RGB used for overlay."""
    pil = Image.fromarray(rgb)
    resized = pil.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    display_rgb = np.asarray(resized)
    tensor = TRANSFORMS(pil).unsqueeze(0).to(DEVICE)
    return tensor, display_rgb

def forward_logits(tensor: torch.Tensor) -> torch.Tensor:
    """Return raw class logits (B, 2). YOLO-cls eval returns (probs, logits)."""
    out = model(tensor)
    if isinstance(out, (tuple, list)):
        return out[1]
    return out

def normalize_cam(cam: np.ndarray) -> np.ndarray:
    cam = np.maximum(cam, 0)
    cam = cam - cam.min()
    denom = cam.max()
    if denom < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)
    return (cam / denom).astype(np.float32)

def upsample_cam(cam: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    return cv2.resize(cam, (size, size), interpolation=cv2.INTER_LINEAR)

def overlay_cam(rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    base = rgb.astype(np.float32)
    blend = (1 - alpha) * base + alpha * heat.astype(np.float32)
    return np.clip(blend, 0, 255).astype(np.uint8)

@dataclass
class Explanation:
    method: str
    cam: np.ndarray
    overlay: np.ndarray
    target_class: int
    target_name: str
    prob_ev: float
    pred_idx: int

class YOLOCAMExplainer:
    """Grad-CAM / Grad-CAM++ / EigenCAM for Ultralytics ClassificationModel."""

    def __init__(self, model: torch.nn.Module, target_layer_name: str):
        self.model = model
        self.target_layer = get_module_by_name(model, target_layer_name)
        self.activations: torch.Tensor | None = None
        self.gradients: torch.Tensor | None = None
        self._fh = self.target_layer.register_forward_hook(self._save_activation)
        self._bh = self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output) -> None:
        self.activations = output if torch.is_tensor(output) else output[0]

    def _save_gradient(self, module, grad_input, grad_output) -> None:
        g = grad_output[0] if isinstance(grad_output, tuple) else grad_output
        self.gradients = g

    def close(self) -> None:
        self._fh.remove()
        self._bh.remove()

    def _forward(self, tensor: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        self.model.zero_grad(set_to_none=True)
        logits = forward_logits(tensor)
        probs = torch.softmax(logits, dim=1)
        return logits, probs

    def predict(self, tensor: torch.Tensor) -> tuple[int, float, torch.Tensor]:
        with torch.no_grad():
            logits = forward_logits(tensor)
            probs = torch.softmax(logits, dim=1)[0]
        pred_idx = int(torch.argmax(probs).item())
        return pred_idx, float(probs[EV_INDEX].item()), probs

    def gradcam(self, tensor: torch.Tensor, target_class: int, plusplus: bool = False) -> np.ndarray:
        tensor = tensor.clone().detach().requires_grad_(True)
        logits, _ = self._forward(tensor)
        score = logits[0, target_class]
        score.backward(retain_graph=False)

        assert self.activations is not None and self.gradients is not None
        acts = self.activations
        grads = self.gradients

        if plusplus:
            grads_pos = F.relu(grads)
            alpha_num = grads_pos.pow(2)
            alpha_den = 2 * grads_pos.pow(2) + (acts * grads_pos.pow(3)).sum(dim=(2, 3), keepdim=True)
            alpha = alpha_num / (alpha_den + 1e-8)
            weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)
        else:
            weights = grads.mean(dim=(2, 3), keepdim=True)

        cam = (weights * acts).sum(dim=1).squeeze(0)
        cam = F.relu(cam).detach().cpu().numpy()
        return normalize_cam(upsample_cam(cam))

    def eigencam(self, tensor: torch.Tensor) -> np.ndarray:
        with torch.no_grad():
            _ = self._forward(tensor)
        assert self.activations is not None
        acts = self.activations.detach().cpu().numpy()[0]
        c, h, w = acts.shape
        flat = acts.reshape(c, -1).T
        flat = flat - flat.mean(axis=0, keepdims=True)
        _, _, vt = np.linalg.svd(flat, full_matrices=False)
        cam = (flat @ vt[0]).reshape(h, w)
        cam = np.abs(cam)
        return normalize_cam(upsample_cam(cam))

def occlusion_map(
    rgb: np.ndarray,
    target_class: int,
    patch: int = OCCLUSION_PATCH,
    stride: int = OCCLUSION_STRIDE,
) -> np.ndarray:
    """Sensitivity map: baseline score − score when a gray patch occludes the region."""
    tensor, display_rgb = preprocess_rgb(rgb)
    with torch.no_grad():
        base_logits = forward_logits(tensor)
        base_prob = float(torch.softmax(base_logits, dim=1)[0, target_class].item())

    h, w = display_rgb.shape[:2]
    heat = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)
    gray = np.full((patch, patch, 3), 127, dtype=np.uint8)

    for y in range(0, h - patch + 1, stride):
        for x in range(0, w - patch + 1, stride):
            occluded = display_rgb.copy()
            occluded[y : y + patch, x : x + patch] = gray
            t, _ = preprocess_rgb(occluded)
            with torch.no_grad():
                logits = forward_logits(t)
                prob = float(torch.softmax(logits, dim=1)[0, target_class].item())
            drop = base_prob - prob
            heat[y : y + patch, x : x + patch] += drop
            counts[y : y + patch, x : x + patch] += 1.0

    counts = np.maximum(counts, 1.0)
    heat = heat / counts
    return normalize_cam(heat)

explainer = YOLOCAMExplainer(model, TARGET_LAYER_NAME)
print("Explainer ready.")


In [ ]:
def explain_image(
    path: str | Path,
    target_class: int | None = None,
    run_occlusion: bool = True,
) -> tuple[np.ndarray, dict[str, Explanation]]:
    """Compute multi-method explanations for one image.

    If target_class is None, explain the model's predicted class.
    Also always compute an EV-targeted Grad-CAM (useful for EV-vs-ICE contrast).
    """
    rgb = load_rgb(path)
    tensor, display_rgb = preprocess_rgb(rgb)
    pred_idx, prob_ev, _ = explainer.predict(tensor)
    if target_class is None:
        target_class = pred_idx
    target_name = IDX_TO_CLASS[target_class]

    results: dict[str, Explanation] = {}

    for method, cam in [
        ("Grad-CAM", explainer.gradcam(tensor, target_class, plusplus=False)),
        ("Grad-CAM++", explainer.gradcam(tensor, target_class, plusplus=True)),
        ("EigenCAM", explainer.eigencam(tensor)),
    ]:
        results[method] = Explanation(
            method=method,
            cam=cam,
            overlay=overlay_cam(display_rgb, cam),
            target_class=target_class,
            target_name=target_name,
            prob_ev=prob_ev,
            pred_idx=pred_idx,
        )

    cam_ev = explainer.gradcam(tensor, EV_INDEX, plusplus=False)
    results["Grad-CAM (EV logit)"] = Explanation(
        method="Grad-CAM (EV logit)",
        cam=cam_ev,
        overlay=overlay_cam(display_rgb, cam_ev),
        target_class=EV_INDEX,
        target_name="ev",
        prob_ev=prob_ev,
        pred_idx=pred_idx,
    )

    if run_occlusion:
        cam_occ = occlusion_map(rgb, target_class)
        results["Occlusion"] = Explanation(
            method="Occlusion",
            cam=cam_occ,
            overlay=overlay_cam(display_rgb, cam_occ),
            target_class=target_class,
            target_name=target_name,
            prob_ev=prob_ev,
            pred_idx=pred_idx,
        )

    return display_rgb, results

def plot_explanation_panel(
    display_rgb: np.ndarray,
    results: dict[str, Explanation],
    title: str,
    save_path: Path | None = None,
) -> None:
    methods = list(results.keys())
    n = 1 + len(methods)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.0))
    if n == 1:
        axes = [axes]

    axes[0].imshow(display_rgb)
    axes[0].set_title("Input")
    axes[0].axis("off")

    for ax, method in zip(axes[1:], methods):
        ax.imshow(results[method].overlay)
        ax.set_title(method)
        ax.axis("off")

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 5) Select paper-ready examples

We pick high-confidence, visually informative cases from each outcome bucket:

1. **TP_ev** — correct EV (what the model trusts for EV)
2. **TN_ice** — correct ICE (what suppresses the EV decision)
3. **FP_ice_as_ev** — ICE called EV (failure mode / spurious cues)
4. **FN_ev_as_ice** — EV called ICE (missed EV cues)
5. **Borderline** — correct but low-confidence (near decision boundary)

In [ ]:
def pick_examples(df: pd.DataFrame, n: int = N_PER_BUCKET) -> dict[str, pd.DataFrame]:
    picks: dict[str, pd.DataFrame] = {}

    for bucket in ["TP_ev", "TN_ice", "FP_ice_as_ev", "FN_ev_as_ice"]:
        sub = df[df["bucket"] == bucket].copy()
        if sub.empty:
            picks[bucket] = sub
            continue
        sub = sub.sort_values("confidence", ascending=False).head(n)
        picks[bucket] = sub.reset_index(drop=True)

    borderline = df[df["correct"]].copy()
    borderline = borderline.sort_values("confidence", ascending=True).head(n)
    picks["borderline_correct"] = borderline.reset_index(drop=True)
    return picks

example_sets = pick_examples(analysis_df, N_PER_BUCKET)

summary_rows = []
for name, sub in example_sets.items():
    summary_rows.append(
        {
            "set": name,
            "n_available": int((analysis_df["bucket"] == name).sum())
            if name in analysis_df["bucket"].unique()
            else (len(analysis_df[analysis_df["correct"]]) if name.startswith("borderline") else 0),
            "n_selected": len(sub),
            "mean_conf": float(sub["confidence"].mean()) if len(sub) else np.nan,
            "mean_prob_ev": float(sub["prob_ev"].mean()) if len(sub) else np.nan,
        }
    )

display(pd.DataFrame(summary_rows))

for name, sub in example_sets.items():
    print(f"\n=== {name} ===")
    if sub.empty:
        print("(none)")
        continue
    display(
        sub[["relative_path", "gt_label", "pred_label", "prob_ev", "confidence"]].round(4)
    )


## 6) Multi-method explanation galleries

Each row is one test image. Columns: input, Grad-CAM, Grad-CAM++, EigenCAM, Grad-CAM on the **EV logit**, and Occlusion.

> Occlusion is slower (sliding gray patches). Set `run_occlusion=False` below if you want a quick pass first.

In [ ]:
RUN_OCCLUSION = True

BUCKET_TITLES = {
    "TP_ev": "True Positive EV — correct EV prediction",
    "TN_ice": "True Negative ICE — correct ICE prediction",
    "FP_ice_as_ev": "False Positive — ICE predicted as EV",
    "FN_ev_as_ice": "False Negative — EV predicted as ICE",
    "borderline_correct": "Borderline correct — low-confidence but right",
}

def explain_bucket(bucket: str, run_occlusion: bool = RUN_OCCLUSION) -> None:
    sub = example_sets[bucket]
    print(f"\n{'=' * 72}\n{BUCKET_TITLES[bucket]}  (n={len(sub)})\n{'=' * 72}")
    if sub.empty:
        print("No examples in this bucket.")
        return

    bucket_dir = OUT_DIR / bucket
    bucket_dir.mkdir(parents=True, exist_ok=True)

    for i, row in sub.iterrows():
        path = Path(row["image_path"])
        display_rgb, results = explain_image(
            path,
            target_class=int(row["y_pred"]),
            run_occlusion=run_occlusion,
        )
        title = (
            f"{path.name} | GT={row['gt_label']}  PRED={row['pred_label']}  "
            f"P(EV)={row['prob_ev']:.3f}  conf={row['confidence']:.3f}"
        )
        save_path = bucket_dir / f"{i:02d}_{path.stem}_panel.png" if SAVE_FIGURES else None
        plot_explanation_panel(display_rgb, results, title, save_path=save_path)

        if SAVE_FIGURES:
            cam_path = bucket_dir / f"{i:02d}_{path.stem}_gradcam.png"
            fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))
            axes[0].imshow(display_rgb)
            axes[0].set_title("Input")
            axes[0].axis("off")
            axes[1].imshow(results["Grad-CAM"].overlay)
            axes[1].set_title(f"Grad-CAM → {results['Grad-CAM'].target_name}")
            axes[1].axis("off")
            fig.suptitle(title, fontsize=9)
            fig.tight_layout()
            fig.savefig(cam_path, bbox_inches="tight")
            plt.close(fig)

for bucket in ["TP_ev", "TN_ice", "FP_ice_as_ev", "FN_ev_as_ice", "borderline_correct"]:
    explain_bucket(bucket)


## 7) EV vs ICE contrast panels (paper figure)

Side-by-side Grad-CAM for a confident **EV** and a confident **ICE** prediction, both with heatmaps for the **EV logit** and the **ICE logit**. This makes class-specific focus easier to discuss in the paper.

In [ ]:
def contrast_pair_figure(
    ev_row: pd.Series,
    ice_row: pd.Series,
    save_name: str = "contrast_ev_vs_ice.png",
) -> None:
    rows = [("EV example", ev_row), ("ICE example", ice_row)]
    fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.5))

    for r_idx, (label, row) in enumerate(rows):
        rgb = load_rgb(row["image_path"])
        tensor, display_rgb = preprocess_rgb(rgb)
        cam_ev = explainer.gradcam(tensor, EV_INDEX, plusplus=False)
        cam_ice = explainer.gradcam(tensor, ICE_INDEX, plusplus=False)

        axes[r_idx, 0].imshow(display_rgb)
        axes[r_idx, 0].set_title(
            f"{label}\nGT={row['gt_label']} PRED={row['pred_label']} P(EV)={row['prob_ev']:.2f}"
        )
        axes[r_idx, 0].axis("off")

        axes[r_idx, 1].imshow(overlay_cam(display_rgb, cam_ev))
        axes[r_idx, 1].set_title("Grad-CAM → EV logit")
        axes[r_idx, 1].axis("off")

        axes[r_idx, 2].imshow(overlay_cam(display_rgb, cam_ice))
        axes[r_idx, 2].set_title("Grad-CAM → ICE logit")
        axes[r_idx, 2].axis("off")

    fig.suptitle("Class-contrast Grad-CAM on held-out test crops (YOLO26s)", y=1.01)
    fig.tight_layout()
    out = OUT_DIR / save_name
    if SAVE_FIGURES:
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved {out}")
    plt.show()
    plt.close(fig)

if len(example_sets["TP_ev"]) and len(example_sets["TN_ice"]):
    contrast_pair_figure(example_sets["TP_ev"].iloc[0], example_sets["TN_ice"].iloc[0])
    if len(example_sets["TP_ev"]) > 1 and len(example_sets["TN_ice"]) > 1:
        contrast_pair_figure(
            example_sets["TP_ev"].iloc[1],
            example_sets["TN_ice"].iloc[1],
            save_name="contrast_ev_vs_ice_alt.png",
        )
else:
    print("Need at least one TP_ev and one TN_ice for contrast panels.")


## 8) Aggregate attention summary (optional quantitative support)

Compute Grad-CAM for every test image (predicted class) and summarize **where** heat concentrates using a 3×3 spatial grid. Useful one-liner for the paper: e.g. “EV predictions focus more on upper-front fascia / emblem regions.”

In [ ]:
def spatial_mass(cam: np.ndarray, grid: int = 3) -> np.ndarray:
    """Fraction of CAM energy in each grid cell (row-major)."""
    h, w = cam.shape
    masses = []
    for i in range(grid):
        for j in range(grid):
            y0, y1 = int(i * h / grid), int((i + 1) * h / grid)
            x0, x1 = int(j * w / grid), int((j + 1) * w / grid)
            masses.append(float(cam[y0:y1, x0:x1].sum()))
    masses = np.asarray(masses, dtype=np.float32)
    total = masses.sum()
    return masses / total if total > 1e-8 else masses

GRID_LABELS = [
    "top-left", "top-center", "top-right",
    "mid-left", "mid-center", "mid-right",
    "bot-left", "bot-center", "bot-right",
]

agg_rows = []
for _, row in analysis_df.iterrows():
    tensor, _ = preprocess_rgb(load_rgb(row["image_path"]))
    cam = explainer.gradcam(tensor, int(row["y_pred"]), plusplus=False)
    masses = spatial_mass(cam, grid=3)
    rec = {
        "bucket": row["bucket"],
        "pred_label": row["pred_label"],
        "gt_label": row["gt_label"],
        "prob_ev": row["prob_ev"],
    }
    for lab, m in zip(GRID_LABELS, masses):
        rec[lab] = m
    agg_rows.append(rec)

agg_df = pd.DataFrame(agg_rows)
agg_path = OUT_DIR / "spatial_attention_summary.csv"
agg_df.to_csv(agg_path, index=False)
print(f"Wrote {agg_path}")

mean_by_pred = agg_df.groupby("pred_label")[GRID_LABELS].mean()
print("\nMean Grad-CAM spatial mass by predicted class:")
display(mean_by_pred.round(3))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
for ax, cls in zip(axes, ["ev", "ice"]):
    if cls not in mean_by_pred.index:
        ax.set_visible(False)
        continue
    mat = mean_by_pred.loc[cls].to_numpy().reshape(3, 3)
    im = ax.imshow(mat, cmap="YlOrRd", vmin=0, vmax=max(0.25, mat.max()))
    ax.set_title(f"Mean attention | pred={cls}")
    ax.set_xticks([0, 1, 2], ["L", "C", "R"])
    ax.set_yticks([0, 1, 2], ["T", "M", "B"])
    for (i, j), val in np.ndenumerate(mat):
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Where YOLO26s Grad-CAM concentrates on the test set", y=1.02)
fig.tight_layout()
grid_fig = OUT_DIR / "spatial_attention_grids.png"
if SAVE_FIGURES:
    fig.savefig(grid_fig, bbox_inches="tight")
    print(f"Saved {grid_fig}")
plt.show()
plt.close(fig)


## 9) Notes for the paper

**Suggested talking points** (verify against the figures above):

1. **Correct EVs (TP):** Grad-CAM / Grad-CAM++ often highlight front fascia, grille/closed-grille area, headlight signature, or brand/emblem regions rather than background road/sky.
2. **Correct ICEs (TN):** EV-logit maps should be weak or scattered; ICE-logit maps may focus on traditional grille / lower bumper geometry.
3. **False positives:** Look for ICE vehicles with EV-like lighting, badges, or crop framing that mimics training EVs — evidence of shortcut features.
4. **False negatives:** Missed EVs where the discriminative region is occluded, low-res, or outside the crop.
5. **Method agreement:** When Grad-CAM, EigenCAM, and Occlusion agree on the same region, the explanation is more trustworthy for qualitative claims.

**Outputs saved under** `runs/explain_stage2_yolo26s/`:
- `{bucket}/*_panel.png` — full multi-method panels
- `{bucket}/*_gradcam.png` — compact 2-panel Grad-CAM figures
- `contrast_ev_vs_ice*.png` — class-contrast figure
- `spatial_attention_summary.csv` / `spatial_attention_grids.png` — aggregate attention

Re-run with `N_PER_BUCKET` increased if you need more candidates for figure selection.

In [ ]:
TP_EV_CERTAIN_THRESH = 0.95

explainer = YOLOCAMExplainer(model, TARGET_LAYER_NAME)

tp_certain = (
    analysis_df[
        (analysis_df["bucket"] == "TP_ev")
        & (analysis_df["prob_ev"] >= TP_EV_CERTAIN_THRESH)
    ]
    .sort_values("prob_ev", ascending=False)
    .reset_index(drop=True)
)

print(
    f"True Positive EV with P(EV) >= {TP_EV_CERTAIN_THRESH}: "
    f"{len(tp_certain)} / {(analysis_df['bucket'] == 'TP_ev').sum()} TP_ev"
)
display(
    tp_certain[["relative_path", "gt_label", "pred_label", "prob_ev", "confidence"]].round(6)
)

if tp_certain.empty:
    print("No TP_ev cases meet the certainty threshold.")
else:
    gallery_dir = OUT_DIR / "tp_ev_p_ev_eq_1"
    gallery_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 72}")
    print(f"True Positive EV — P(EV) ≈ 1  (n={len(tp_certain)})")
    print(f"{'=' * 72}")

    for i, row in tp_certain.iterrows():
        path = Path(row["image_path"])
        display_rgb, results = explain_image(
            path,
            target_class=int(row["y_pred"]),
            run_occlusion=RUN_OCCLUSION,
        )
        title = (
            f"{path.name} | GT={row['gt_label']}  PRED={row['pred_label']}  "
            f"P(EV)={row['prob_ev']:.3f}  conf={row['confidence']:.3f}"
        )
        save_path = (
            gallery_dir / f"{i:02d}_{path.stem}_panel.png" if SAVE_FIGURES else None
        )
        plot_explanation_panel(display_rgb, results, title, save_path=save_path)

        if SAVE_FIGURES:
            cam_path = gallery_dir / f"{i:02d}_{path.stem}_gradcam.png"
            fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))
            axes[0].imshow(display_rgb)
            axes[0].set_title("Input")
            axes[0].axis("off")
            axes[1].imshow(results["Grad-CAM"].overlay)
            axes[1].set_title(f"Grad-CAM → {results['Grad-CAM'].target_name}")
            axes[1].axis("off")
            fig.suptitle(title, fontsize=9)
            fig.tight_layout()
            fig.savefig(cam_path, bbox_inches="tight")
            plt.close(fig)

    print(f"Saved panels under: {gallery_dir}" if SAVE_FIGURES else "Figures not saved (SAVE_FIGURES=False).")

explainer.close()
print("Done. Figures are in:", OUT_DIR)
